# HCC–VirTues: private dataset acquisition and first QC
Run this notebook in Google Colab. It keeps images, masks, and patient-level data in the runtime; it does not upload them to GitHub. The notebook only performs acquisition and structural QC. It does **not** run VirTues inference or establish biological findings.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/content/hcc_virtues_private')
ROOT.mkdir(parents=True, exist_ok=True)
print('Private working directory:', ROOT)
print('Runtime:', sys.version)


In [ ]:
# Download the research scripts/templates from the active project branch.
import subprocess
REPO = 'https://github.com/culprit-k/cancer-immunotherapy-robust-discovery.git'
BRANCH = 'hcc-virtues-audit-next-step'
checkout = Path('/content/cancer-immunotherapy-robust-discovery')
if not checkout.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,str(checkout)], check=True)
else:
    subprocess.run(['git','-C',str(checkout),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(checkout),'checkout',BRANCH], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','tifffile','imageio'], check=True)
print('Project checkout:', checkout)


In [ ]:
# Download, verify MD5, and extract the public archive. This can take several minutes.
import hashlib, urllib.request, zipfile
archive = ROOT / 'HCC_ICICohort.zip'
url = 'https://zenodo.org/records/10794904/files/HCC_ICICohort.zip?download=1'
expected_md5 = 'd0b3242740b70c1355f67bc7bd176c6a'
if not archive.exists():
    urllib.request.urlretrieve(url, archive)
h = hashlib.md5()
with archive.open('rb') as f:
    for chunk in iter(lambda: f.read(1024*1024), b''):
        h.update(chunk)
actual_md5 = h.hexdigest()
print('Archive bytes:', archive.stat().st_size, 'MD5:', actual_md5)
if actual_md5 != expected_md5:
    raise RuntimeError(f'MD5 mismatch: expected {expected_md5}, got {actual_md5}. Do not extract/use this archive.')
extract_dir = ROOT / 'extracted'
marker = ROOT / '.extracted_ok'
if not marker.exists():
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(extract_dir)
    marker.write_text('verified and extracted')
print('Verified and extracted to:', extract_dir)


In [ ]:
# Inventory folders and locate the panel CSV without assuming the archive's directory layout.
all_files = [p for p in extract_dir.rglob('*') if p.is_file()]
print('Total files:', len(all_files))
for p in all_files[:80]: print(p.relative_to(extract_dir))
panel_files = [p for p in all_files if p.name.lower() == 'icicohort_panel.csv']
print('\nPanel CSV candidates:', panel_files)
for p in panel_files:
    import pandas as pd
    panel = pd.read_csv(p)
    print('\nPanel file:', p.name, '| shape:', panel.shape)
    print('Columns:', list(panel.columns))
    display(panel.head(30))


In [ ]:
# Inspect TIFF series axes/shapes on a few files. This is a preview, not the full QC.
import tifffile
tiffs = [p for p in all_files if p.suffix.lower() in ('.tif','.tiff')]
print('TIFF files:', len(tiffs))
for p in tiffs[:8]:
    try:
        with tifffile.TiffFile(p) as tf:
            s = tf.series[0]
            print(p.name, '| series axes:', s.axes, '| shape:', s.shape, '| dtype:', s.dtype)
    except Exception as e:
        print(p.name, '| READ ERROR:', repr(e))


## Next gate: do not guess the channel mapping
Use the panel file and the observed TIFF axes to establish the channel order. If the archive stores one TIFF per marker/channel, document that structure explicitly rather than treating it as a multichannel tensor. Resolve marker identifiers and intensity semantics from source metadata; leave uncertain mappings unresolved. Keep the filled map private. Then run the repository QC script with the actual image, cell-mask, and stromal-mask folders. The QC instructions are in `docs/HCC_local_tensor_QC_instructions.md`. Do not run model inference until ROI pairing, mask alignment, marker order, checkpoint exposure, and preprocessing compatibility are reviewed.

In [ ]:
# Print likely image/mask directory candidates to make the QC command easier to configure.
for p in sorted({x.parent for x in all_files if x.suffix.lower() in ('.tif','.tiff','.png','.npy')}, key=str):
    sample = [x.name for x in p.iterdir() if x.is_file()][:3]
    print(p, '| sample files:', sample)
